# fishbehavior on Google Colab: quick start

Runs the whole behavior-labeling pipeline on videos stored in your Google Drive:
catalog, scene setup, tracking, features, labels, (optional) calibration, datasets and plots.

**No GPU needed.** Everything in this stage is classic computer vision and small statistical
models on the CPU (OpenCV, scikit-learn). A normal CPU runtime is enough:
*Runtime > Change runtime type > CPU*.

**Before you start**, put these in a Drive folder (default below: `MyDrive/fishbehavior/`):

- `videos/`: the trial videos (e.g. `F_0042.mp4`);
- `database.xlsx`: the trial workbook;
- `reference.pdf` (optional): the presentation with the reference ethograms, for calibration.

The results go to `MyDrive/fishbehavior/outputs/`, so they survive a disconnect. Running
`all` again continues where it stopped: finished work is skipped. See
`docs/behavior-labeling.md` for the full guide.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Get the code and install it

If the repository is private, use a GitHub personal access token:
`https://<token>@github.com/Fish-Behavior/fish-detection.git`. Never save the token in this notebook.

In [ ]:
!git clone https://github.com/Fish-Behavior/fish-detection.git
%cd fish-detection
!python -m pip install -q -e .

## 3. Point the pipeline at your Drive folders

These replace the `.env` file used on a laptop. Change `DRIVE` if your folder is elsewhere.
Leave `FISH_REFERENCE_PDF` out if you have no reference PDF (calibration is then skipped).

In [ ]:
import os

DRIVE = "/content/drive/MyDrive/fishbehavior"
os.environ["FISH_VIDEO_DIR"] = f"{DRIVE}/videos"
os.environ["FISH_DB_PATH"] = f"{DRIVE}/database.xlsx"
os.environ["FISH_REFERENCE_PDF"] = f"{DRIVE}/reference.pdf"
os.environ["FISH_OUTPUT_DIR"] = f"{DRIVE}/outputs"
os.environ["FISH_WORKERS"] = str(os.cpu_count())  # videos processed in parallel

## 4. Check the setup

Every path should say `[ok]`.

In [ ]:
!python -m fishbehavior check-config

## 5. Catalog: clean the workbook and match videos

Read `outputs/catalog/validation_report.md` for subjects without a video, split recordings, etc.

In [ ]:
!python -m fishbehavior validate

## 6. Try one subject first

Replace `42` with one of your subject numbers (`42`, `0042` and `F_0042` all work). This runs
scene setup, tracking, features, labeling, datasets and plots for that subject, and prints
how long each step took. Look at `outputs/scene/<video>_qa.png` and
`outputs/tracks/<subject>_track_qa.png` before running everything.

The scene-review page (to correct a waterline or region) can't open a local server on Colab.
Use `python -m fishbehavior scene-review --no-serve`, download `outputs/scene/review.html`,
open it on your computer, and put the saved `overrides.yaml` back into `outputs/scene/`.

In [ ]:
!python -m fishbehavior all --subjects 42

## 7. All subjects

Tracking takes the longest (minutes per 20-minute video). If Colab disconnects, run this cell
again: finished subjects are skipped. On the first run with a reference PDF, the `reference`
step writes `outputs/reference/mapping.yaml` for you to fill in (see the guide); calibration
runs on the next `all` after that.

In [ ]:
!python -m fishbehavior all

## 8. Look at the results

- `outputs/datasets/`: `behavior_dataset.xlsx` (one row per subject, with a column guide),
  `behavior_windows.csv` (training manifest), per-second labels, segments;
- `outputs/plots/`: ethograms per group and bar charts per state;
- a review video of one subject: the next cell (`--start`/`--end` in seconds).

In [ ]:
!python -m fishbehavior plot --subjects 42 --overlay --start 0 --end 120